# Azure Run-All Notebook - Two-Stage Trading Product

Use this notebook on an Azure Machine Learning compute instance when you want the closest thing to: open notebook -> Run All -> get metrics, CSVs, and plots.

Recommended workflow:
1. Start an Azure ML compute instance.
2. Clone this repo into your workspace files.
3. Put `data/spy_1min_clean.csv` in place.
4. Open this notebook.
5. Leave `INSTALL_REQUIREMENTS = True` for the first run, then switch it to `False` later.
6. Press **Run All**.

This notebook:
- installs repo dependencies into the current notebook kernel
- runs `run_trading_product.py`
- loads the saved JSON/CSV outputs
- renders the key execution and equity plots inline

Note: the full rolling-window Stage 1 dataset is much heavier than the earlier selected-slice version. A GPU-backed compute instance is strongly preferred for faster runs.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'run_trading_product.py').exists() else NOTEBOOK_DIR.parent
if not (REPO_ROOT / 'run_trading_product.py').exists():
    raise FileNotFoundError('Could not locate repo root containing run_trading_product.py')

os.chdir(REPO_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / '.mplconfig'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Python executable: {sys.executable}')


In [ ]:
INSTALL_REQUIREMENTS = True
CONFIG_OVERRIDE = 'configs/azure_compute.yaml'
RUN_PIPELINE = True

def _deep_merge(base: dict, override: dict) -> dict:
    merged = dict(base)
    for key, value in override.items():
        if key in merged and isinstance(merged[key], dict) and isinstance(value, dict):
            merged[key] = _deep_merge(merged[key], value)
        else:
            merged[key] = value
    return merged

with open(REPO_ROOT / 'configs' / 'config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

if CONFIG_OVERRIDE:
    override_path = Path(CONFIG_OVERRIDE)
    if not override_path.is_absolute():
        override_path = REPO_ROOT / override_path
    with open(override_path, encoding='utf-8') as f:
        cfg = _deep_merge(cfg, yaml.safe_load(f))

ACTIVE_PATTERN = cfg['labeling']['active_pattern']
DATA_PATH = REPO_ROOT / cfg['data_source']['file_path']
PRODUCT_ROOT = REPO_ROOT / 'outputs' / 'product' / ACTIVE_PATTERN

print(f'Active pattern: {ACTIVE_PATTERN}')
print(f'Data path: {DATA_PATH}')
print(f'Output root: {PRODUCT_ROOT}')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset: {DATA_PATH}. Upload or copy data/spy_1min_clean.csv before running.'
    )


In [ ]:
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--extra-index-url',
            'https://download.pytorch.org/whl/cu124',
            '-r',
            'requirements.txt',
        ],
        check=True,
    )
else:
    print('Skipping dependency install. Set INSTALL_REQUIREMENTS = True on a fresh compute instance.')


In [ ]:
cmd = [sys.executable, 'run_trading_product.py']
if CONFIG_OVERRIDE:
    cmd.extend(['--config-override', CONFIG_OVERRIDE])

print('Running command:')
print(' '.join(cmd))

if RUN_PIPELINE:
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
else:
    print('Skipping pipeline execution. Set RUN_PIPELINE = True to execute the full product run.')


In [ ]:
summary_path = PRODUCT_ROOT / 'summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Expected summary file not found: {summary_path}')

summary = json.loads(summary_path.read_text(encoding='utf-8'))

display(Markdown(f"## Run Summary for `{summary['pattern']}`"))

if 'rule_labeled_anchor_count' in summary:
    display(Markdown('### Rule-Labeled Anchors'))
    display(pd.DataFrame([summary['rule_labeled_anchor_count']]))

if 'stage1_dataset' in summary:
    display(Markdown('### Stage 1 Dataset'))
    stage1_dataset_df = pd.json_normalize(summary['stage1_dataset'])
    display(stage1_dataset_df)

display(Markdown('### Stage 1 Test Metrics'))
display(pd.DataFrame([summary['stage1_test']]))

display(Markdown('### Stage 2 Test Metrics'))
display(pd.DataFrame([summary['stage2_test']]))

display(Markdown('### Stage 1 Only vs Stage 1 + Stage 2'))
comparison_path = PRODUCT_ROOT / 'stage2' / 'profitability_comparison.csv'
display(pd.read_csv(comparison_path))

paper_summary_path = PRODUCT_ROOT / 'paper_replay' / 'paper_summary.json'
if paper_summary_path.exists() and paper_summary_path.stat().st_size > 0:
    paper_summary = json.loads(paper_summary_path.read_text(encoding='utf-8'))
    display(Markdown('### Paper Replay'))
    display(pd.DataFrame([paper_summary]))

trade_candidates_path = PRODUCT_ROOT / 'stage2' / 'test_trade_candidates.csv'
if trade_candidates_path.exists():
    display(Markdown('### Sample Test Trade Candidates'))
    display(pd.read_csv(trade_candidates_path).head(20))


In [ ]:
def show_image(title: str, path: Path) -> None:
    if path.exists() and path.stat().st_size > 0:
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path)))
    else:
        print(f'Skipping missing or empty image: {path}')

show_image('Stage 2 Test Execution Overview', PRODUCT_ROOT / 'stage2' / 'test_execution_overview.png')
show_image('Stage 2 Test Execution Panels', PRODUCT_ROOT / 'stage2' / 'test_execution_panels.png')
show_image('Paper Replay Equity Curve', PRODUCT_ROOT / 'paper_replay' / 'paper_equity.png')
show_image('Paper Replay Execution Overview', PRODUCT_ROOT / 'paper_replay' / 'paper_execution_overview.png')
show_image('Paper Replay Execution Panels', PRODUCT_ROOT / 'paper_replay' / 'paper_execution_panels.png')
